In [19]:
import numpy as np
import heapq
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks 
from scipy.interpolate import interp1d

from model import VectorNet, CompareNet
from utils import calculate_position_stats_numpy

## 1. Load matching network

In [ ]:
f_model = VectorNet()
c_model = CompareNet()

def inference(f_model, c_model, vector_1, vector_2):
    """
    Description: This function is used to perform inference on the model
    f_model: feature extraction model
    c_model: comparison model
    vector_1: input vector 1
    vector_2: input vector 2
    """
    f_model.eval()
    c_model.eval()
    feature_1 = f_model(vector_1)
    feature_2 = f_model(vector_2)
    dis = c_model(torch.stack((feature_1, feature_2), 1))
    
    return dis

## 2. Traditional baseline methods
The following section implements several traditional computational methods as baseline comparisons against WisePanda. These methods (as described below) represent different approaches to vector similarity measurement and sequence alignment, serving as benchmarks to evaluate WisePanda's performance advantages.  
(1) DTW (1994) Using dynamic time warping to find patterns in time series.  
(2) Fast Matching Method (2003) Curve matching using the fast marching method.  
(3) Scale Invariant Signature (SIS) (1998) Differential and numerically invariant signature curves applied to object recognition.  
(4) Event-DTW (2020) An Improved Dynamic Time Warping Algorithm for Aligning Biomedical Signals of Nonuniform Sampling Frequencies  
(5) Drop-DTW (2021) Aligning Common Signal Between Sequences While Dropping Outliers.  
(6) Modified COW (2022) Modification of Correlation Optimized Warping Method for Position Alignment of Condition Measurements of Linear Assets.    
(7) Two-stage DTW（2025）Two-stage dynamic time warping for intelligent fault detection of rotating machinery under variable speed.  

## 3. Dataset loading and validation setup
This section loads manually curated datasets that provide ground truth pairings for algorithm performance validation. The datasets consist of expertly matched ancient document fragments with known correspondences.  
(1) Core Datasets
&nbsp;&nbsp;&nbsp;&nbsp;**vector_real_118_patch.npy**: Bamboo slip dataset (Bamboo236) containing 118 pairs of matched bamboo fragments.<br>
(2) Extended Datasets (Interference Data)
&nbsp;&nbsp;&nbsp;&nbsp;**interference_data_bamboo.npy**: Additional unmatched fragments used to expand the candidate pool (Bamboo1350).


In [ ]:
def get_top_k_accuracy(k, dis_map, direction):
    """
    Description: This function is used to calculate the top k accuracy of the model
    k: top k
    dis_map: distance map
    direction: "longitudinal" or "transverse"
    """
    # calculate the top k accuracy of the model
    top_k = 0
    length = len(dis_map)
    length = 118
    gather_list_transverse = []
    for i in range(length):
        dis_list = []
        if direction == "longitudinal":
           dis_list = [x[i] for x in dis_map]
        else: 
            dis_list = dis_map[i].copy()
        dis_list.sort()

        top_index = dis_list.index(dis_map[i][i])
        gather_list_transverse.append(top_index+1)
    for j in gather_list_transverse:
        if j <= k:
            top_k += 1
     
    return top_k

def get_topk_accuracy_with_extend(extend_number, method, parameter=None, f_model_pth=None, c_model_pth=None):
    """
    Description: This function is used to calculate the top k accuracy of the model with extended data
    extend_number: number of extended data
    method: method name
    parameter: method parameter
    f_model_pth: path of the feature extraction model
    c_model_pth: path of the comparison model
    """
    # data loading
    real_vectors = np.load("dataset/vector_real_118_patch.npy")
    real_vectors = real_vectors[:, 2:4, :].astype(np.float32)
    interference_data = np.load("dataset/interference_data_bamboo.npy")
    interference_data = interference_data[:extend_number]
    
    # data combination
    top_origin_list = []
    bottom_list = []
    for i in real_vectors:
        top_origin_list.append(i[0])
        bottom_list.append(i[1])
    for i in interference_data:
        top_origin_list.append(i)
        bottom_list.append(i)

    # model loading
    f_model_pth = f_model_pth if f_model_pth else 'models/f_model.pth'
    c_model_pth = c_model_pth if c_model_pth else 'models/c_model.pth'

    f_model.load_state_dict(torch.load(f_model_pth))
    c_model.load_state_dict(torch.load(c_model_pth))    

    length = len(top_origin_list)
    dis_map = []
    for i in range(length):
        dis_list = []
        for j in range(length):
            if "wisepanda" in method or method == "GAN" or method == "seriesGAN" or method == "Diffusion" or method == "Diffusion-TS":
                # wisepanda
                v1 = top_origin_list[i].reshape(1, 1, -1)
                v2 = bottom_list[j].reshape(1, 1, -1)

                v1 = v1 - v1.min()  
                v2 = v2 - v2.min()  

                v1 = v1 / (v1.max() + 1e-6) 
                v2 = v2 / (v2.max() + 1e-6)

                v1 = torch.tensor(v1, dtype=torch.float32)
                v2 = torch.tensor(v2, dtype=torch.float32)

                dis = inference(f_model, c_model, v1, v2)
                dis_list.append(dis.item())
            else:   
                # Implementation details refer to original papers and source code of each curve matching method
                continue  
        dis_map.append(dis_list)
    k_list = [1, 5, 10, 20, 50, 100]
    result_list = []
    for i in k_list:
        k = get_top_k_accuracy(i, dis_map, "longitudinal")
        k_ = get_top_k_accuracy(i, dis_map, "transverse")
        result_list.append(round((k+k_)/236, 4))

    return dis_map, result_list

## 4. Performance evaluation results
**Performance testing**

The following section evaluates each method's performance under baseline conditions without and with interference data (extend_number = 0 or 1114). Results are reported for Top-k accuracy where k = [1, 5, 10, 20, 50, 100], representing the accuracy when the correct match appears within the top k candidates, totally test for 3 times to get mean and std.

### 4.1 Wisepanda result

In [ ]:
# wisepanda
valid_info = ['normal', 'wisepanda']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1000, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("wisepanda in normal (236) & means:", means)
print("wisepanda in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("wisepanda in normal (1350) & means:", extend_means)
print("wisepanda in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
wisepanda in normal (236) & means: [13.42, 37.15, 50.85, 66.81, 91.81, 100.0]
wisepanda in normal (236) & stds: [1.07, 1.76, 1.53, 2.89, 2.01, 0.0]
result in extend data (bamboo1350):
wisepanda in normal (1350) & means: [5.37, 16.53, 25.42, 37.57, 52.54, 64.4]
wisepanda in normal (1350) & stds: [1.76, 1.7, 1.12, 0.88, 0.85, 2.24]


### 4.1 Generated model results

#### 4.1.1 Generated model results in normal condition

In [ ]:
# GAN
valid_info = ['normal', 'GAN']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results   
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("GAN in normal (236) & means:", means)
print("GAN in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("GAN in normal (1350) & means:", extend_means)
print("GAN in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
GAN in normal (236) & means: [4.52, 17.94, 32.06, 54.52, 81.36, 96.61]
GAN in normal (236) & stds: [1.07, 1.07, 3.66, 2.45, 0.85, 0.73]
result in extend data (bamboo1350):
GAN in normal (1350) & means: [0.56, 4.24, 7.77, 14.69, 26.27, 41.95]
GAN in normal (1350) & stds: [0.65, 0.85, 1.71, 3.6, 5.73, 8.35]


In [ ]:
# seriesGAN
valid_info = ['normal', 'seriesGAN']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)

means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("seriesGAN in normal (236) & means:", means)
print("seriesGAN in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("seriesGAN in normal (1350) & means:", extend_means)
print("seriesGAN in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
seriesGAN in normal (236) & means: [4.38, 19.49, 32.49, 52.97, 83.62, 96.75]
seriesGAN in normal (236) & stds: [2.69, 4.04, 4.11, 3.88, 1.07, 0.65]
result in extend data (bamboo1350):
seriesGAN in normal (1350) & means: [1.69, 6.78, 12.43, 21.33, 38.28, 49.15]
seriesGAN in normal (1350) & stds: [1.47, 0.73, 1.91, 2.18, 3.53, 4.78]


In [ ]:
# Diffusion
valid_info = ['normal', 'Diffusion']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("Diffusion in normal (236) & means:", means)
print("Diffusion in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion in normal (1350) & means:", extend_means)
print("Diffusion in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion in normal (236) & means: [3.39, 10.03, 19.63, 34.75, 70.48, 96.05]
Diffusion in normal (236) & stds: [0.42, 0.88, 2.55, 0.85, 5.52, 0.98]
result in extend data (bamboo1350):
Diffusion in normal (1350) & means: [0.42, 2.26, 3.53, 5.23, 12.71, 21.47]
Diffusion in normal (1350) & stds: [0.43, 0.49, 1.07, 1.07, 1.7, 2.59]


In [ ]:
# Diffusion-TS
valid_info = ['normal', 'Diffusion-TS']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("Diffusion-TS in normal (236) & means:", means)
print("Diffusion-TS in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion-TS in normal (1350) & means:", extend_means)
print("Diffusion-TS in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion-TS in normal (236) & means: [3.39, 14.83, 27.26, 45.91, 80.37, 98.73]
Diffusion-TS in normal (236) & stds: [1.47, 1.27, 2.82, 2.82, 3.55, 0.85]
result in extend data (bamboo1350):
Diffusion-TS in normal (1350) & means: [1.55, 4.52, 8.76, 15.54, 27.97, 43.08]
Diffusion-TS in normal (1350) & stds: [1.3, 1.36, 2.09, 2.72, 1.12, 2.89]


#### 4.1.1 Generated model results in perturbation condition

In [ ]:
# GAN
valid_info = ['perturbation', 'GAN']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("GAN in perturbation (236) & means:", means)
print("GAN in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("GAN in perturbation (1350) & means:", extend_means)
print("GAN in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
GAN in perturbation (236) & means: [4.1, 15.4, 27.68, 49.72, 81.64, 96.89]
GAN in perturbation (236) & stds: [0.65, 1.07, 2.89, 4.27, 1.29, 0.24]
result in extend data (bamboo1350):
GAN in perturbation (1350) & means: [0.99, 4.94, 8.05, 12.15, 23.45, 39.55]
GAN in perturbation (1350) & stds: [0.65, 0.65, 1.52, 2.82, 2.59, 4.69]


In [ ]:
# seriesGAN
valid_info = ['perturbation', 'seriesGAN']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("seriesGAN in perturbation (236) & means:", means)
print("seriesGAN in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("seriesGAN in perturbation (1350) & means:", extend_means)
print("seriesGAN in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
seriesGAN in perturbation (236) & means: [3.81, 21.89, 36.44, 56.21, 78.96, 96.47]
seriesGAN in perturbation (236) & stds: [0.43, 1.3, 1.27, 4.69, 1.71, 0.65]
result in extend data (bamboo1350):
seriesGAN in perturbation (1350) & means: [0.84, 6.21, 12.43, 19.07, 36.58, 50.99]
seriesGAN in perturbation (1350) & stds: [0.73, 3.55, 2.82, 3.47, 3.4, 6.59]


In [ ]:
# Diffusion
valid_info = ['perturbation', 'Diffusion']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("Diffusion in perturbation (236) & means:", means)
print("Diffusion in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion in perturbation (1350) & means:", extend_means)
print("Diffusion in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion in perturbation (236) & means: [2.68, 11.16, 22.32, 41.25, 77.54, 96.61]
Diffusion in perturbation (236) & stds: [1.49, 2.18, 3.12, 3.43, 1.85, 0.42]
result in extend data (bamboo1350):
Diffusion in perturbation (1350) & means: [0.56, 1.98, 4.8, 9.18, 22.32, 35.17]
Diffusion in perturbation (1350) & stds: [0.98, 1.49, 1.29, 2.13, 4.88, 4.04]


In [ ]:
# Diffusion-TS
valid_info = ['perturbation', 'Diffusion-TS']
means_stds_list = []
extend_means_stds_list = []

for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])

print("result in original data (bamboo236):")
print("Diffusion-TS in perturbation (236) & means:", means)
print("Diffusion-TS in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion-TS in perturbation (1350) & means:", extend_means)
print("Diffusion-TS in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion-TS in perturbation (236) & means: [6.5, 20.62, 33.05, 50.57, 79.8, 97.74]
Diffusion-TS in perturbation (236) & stds: [1.3, 3.94, 3.47, 5.72, 1.29, 0.88]
result in extend data (bamboo1350):
Diffusion-TS in perturbation (1350) & means: [2.12, 6.5, 11.16, 17.94, 31.92, 46.04]
Diffusion-TS in perturbation (1350) & stds: [0.43, 0.49, 2.44, 3.82, 3.42, 3.29]


### 4.2 Curve matching method results

In [ ]:
# DTW (Dynamic Time Warping) (1994) 
parameter_list = [0.8, 1, 1.2]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("dynamic_time_warping(236) & means:", means)
print("dynamic_time_warping(236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("dynamic_time_warping (1350) & means:", extend_means)
print("dynamic_time_warping (1350) & stds:", extend_stds)

result in original data (bamboo236):
dynamic_time_warping(236) & means: [8.05, 22.88, 31.36, 48.73, 74.72, 94.63]
dynamic_time_warping(236) & stds: [0.42, 0.43, 0.43, 1.47, 2.0, 0.89]
result in extended data (bamboo1114):
dynamic_time_warping (1350) & means: [3.81, 10.73, 14.83, 18.93, 32.48, 48.02]
dynamic_time_warping (1350) & stds: [0.43, 0.65, 0.0, 0.25, 0.88, 0.25]


In [ ]:
# Fast Matching Method (2003)
parameter_list = [0.8, 1, 1.2]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="fast_matching", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="fast_matching", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("fast_matching (236) & means:", means)
print("fast_matching (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("fast_matching (1350) & means:", extend_means)
print("fast_matching (1350) & stds:", extend_stds)

result in original data (bamboo236):
fast_matching (236) & means: [8.9, 18.64, 26.41, 37.71, 65.25, 94.35]
fast_matching (236) & stds: [0.85, 0.0, 0.65, 1.53, 1.7, 0.24]
result in extended data (bamboo1114):
fast_matching (1350) & means: [3.96, 11.01, 14.41, 16.39, 23.45, 38.28]
fast_matching (1350) & stds: [0.49, 0.73, 0.73, 0.25, 1.07, 0.65]


In [ ]:
# Scale Invariant Signature (SIS) (1998)
parameter_list = ['l1', 'l2', 'correlation']
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="sis", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="sis", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("scale_invariant_signatures(236) & means:", means)
print("scale_invariant_signatures(236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("scale_invariant_signatures (1350) & means:", extend_means)
print("scale_invariant_signatures (1350) & stds:", extend_stds)

result in original data (bamboo236):
scale_invariant_signatures(236) & means: [10.45, 21.47, 29.52, 44.35, 71.47, 94.07]
scale_invariant_signatures(236) & stds: [0.24, 0.98, 2.45, 4.65, 2.45, 0.0]
result in extended data (bamboo1114):
scale_invariant_signatures (1350) & means: [7.34, 14.4, 18.08, 22.18, 32.63, 44.35]
scale_invariant_signatures (1350) & stds: [0.25, 1.47, 1.71, 1.96, 3.67, 4.65]


In [ ]:
# event_dtw (2020)
parameter_list = [1, 3, 5]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="event_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="event_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)

means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("event_dtw (2020) (236) & means:", means)
print("event_dtw (2020) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("event_dtw (2020) (1350) & means:", extend_means)
print("event_dtw (2020) (1350) & stds:", extend_stds)

result in original data (bamboo236):
event_dtw (2020) (236) & means: [7.63, 24.01, 31.35, 48.45, 75.42, 94.63]
event_dtw (2020) (236) & stds: [0.0, 0.65, 0.73, 1.07, 1.12, 0.25]
result in extended data (bamboo1114):
event_dtw (2020) (1350) & means: [3.95, 12.71, 16.53, 21.19, 35.03, 49.72]
event_dtw (2020) (1350) & stds: [0.25, 0.73, 0.73, 1.27, 1.37, 0.98]


In [ ]:
# drop_dtw (2021)
parameter_list = [0.1, 0.5, 1.0]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="drop_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="drop_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("drop_dtw (2021) (236) & means:", means)
print("drop_dtw (2021) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("drop_dtw (2021) (1350) & means:", extend_means)
print("drop_dtw (2021) (1350) & stds:", extend_stds)

result in original data (bamboo236):
drop_dtw (2021) (236) & means: [8.34, 23.02, 31.64, 49.29, 75.56, 94.92]
drop_dtw (2021) (236) & stds: [1.22, 0.65, 0.48, 0.25, 0.65, 0.0]
result in extended data (bamboo1114):
drop_dtw (2021) (1350) & means: [3.96, 9.89, 14.69, 19.07, 32.2, 47.88]
drop_dtw (2021) (1350) & stds: [0.49, 0.49, 0.24, 0.0, 0.73, 0.0]


In [ ]:
# modified_cow (2022）
parameter_list = [8, 10, 12]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="modified_cow", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="modified_cow", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("modified_cow (2022) (236) & means:", means)
print("modified_cow (2022) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("modified_cow (2022) (1350) & means:", extend_means)
print("modified_cow (2022) (1350) & stds:", extend_stds)

result in original data (bamboo236):
modified_cow (2022) (236) & means: [7.63, 24.01, 37.99, 56.64, 84.47, 98.59]
modified_cow (2022) (236) & stds: [0.85, 0.24, 1.71, 1.22, 0.49, 0.65]
result in extended data (bamboo1114):
modified_cow (2022) (1350) & means: [1.41, 5.79, 10.17, 15.54, 26.41, 38.98]
modified_cow (2022) (1350) & stds: [0.65, 0.88, 1.12, 0.25, 1.29, 1.53]


In [ ]:
# two_stage_dtw (2025)
parameter_list = [3, 5, 7]
means_stds_list = []
extend_means_stds_list = []

for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="two_stage_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="two_stage_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
    
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("two_stage_dtw (2025) (236) & means:", means)
print("two_stage_dtw (2025) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("two_stage_dtw (2025) (1350) & means:", extend_means)
print("two_stage_dtw (2025) (1350) & stds:", extend_stds)

result in original data (bamboo236):
two_stage_dtw (2025) (236) & means: [7.63, 23.31, 30.65, 49.44, 74.58, 95.34]
two_stage_dtw (2025) (236) & stds: [0.0, 0.73, 0.24, 0.25, 0.0, 0.0]
result in extended data (bamboo1114):
two_stage_dtw (2025) (1350) & means: [4.1, 11.16, 14.97, 19.21, 33.05, 49.15]
two_stage_dtw (2025) (1350) & stds: [0.25, 0.49, 0.24, 0.24, 0.42, 0.43]
